# Week 4 — Lab 1: Introduction to Large Language Models

Welcome to Week 4! This week is all about modern AI — the technology behind tools like ChatGPT, Claude, and Gemini. Before you can prompt these models effectively (next lab) or build applications with them (later labs), you need to understand what a Large Language Model (LLM) actually is, how it processes text, and what its real capabilities and limits are.

This lab uses free, open-source models from Hugging Face — no API key or paid account required. Everything runs directly in this notebook.

**After this lab you will be able to:**
- Explain what an LLM is and how it generates text, at a working level of detail
- Understand tokenization: how text is broken into pieces a model can process
- Understand context windows and why they matter
- Understand sampling parameters (temperature, top-p) and how they change a model's output
- Recognize the real limitations of LLMs (hallucination, knowledge cutoffs, bias)

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.
- Read every explanation carefully — this lab is conceptually dense, and later labs (Prompt Engineering, RAG, AI Agents) all build directly on these ideas.

## Setup

We'll use Hugging Face's `transformers` library, which gives free access to open-source language models. Run the cell below once to install what we need.

In [ ]:
!pip install -q transformers torch

## What Is a Large Language Model?

A Large Language Model (LLM) is a neural network trained on massive amounts of text to predict the next word (technically, the next *token*) in a sequence, given everything that came before it. That's it, at its core — an LLM is fundamentally a very sophisticated next-token predictor.

What makes this powerful is scale. Models like GPT-4 or Claude are trained on hundreds of billions of words and have billions of internal parameters (adjustable weights). At that scale, predicting "the next token" well enough requires the model to implicitly learn grammar, facts, reasoning patterns, and even some problem-solving strategies — not because anyone programmed those rules in, but because they are useful for the underlying task of prediction.

**Key distinction:** an LLM does not "look things up" in a database when it answers you. It generates each token based on learned statistical patterns from training. This is exactly why LLMs can be extremely fluent and also confidently wrong (more on this later — this is called hallucination).

## Tokenization: How Models Read Text

Models don't see words — they see **tokens**, which are chunks of text (sometimes whole words, sometimes sub-word pieces, sometimes single characters). Every model has a fixed vocabulary of tokens it was trained with, and any input text must first be converted into a sequence of token IDs from that vocabulary.

Let's see this in action using GPT-2's tokenizer.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

text = "Machine learning models process tokens, not words."
tokens = tokenizer.tokenize(text)
token_ids = tokenizer.encode(text)

print("Original text:", text)
print("Tokens:", tokens)
print("Number of tokens:", len(tokens))
print("Token IDs:", token_ids)

Notice that some words became multiple tokens, and some tokens include a leading space (shown as `Ġ` in GPT-2's tokenizer). This is why token count is not the same as word count — as a rough rule of thumb, 1 token is about 4 characters or 0.75 words in English.

In [ ]:
# Try a few different texts and compare token counts
examples = [
    "Hello!",
    "Supercalifragilisticexpialidocious",
    "The quick brown fox jumps over the lazy dog.",
]

for ex in examples:
    n_tokens = len(tokenizer.encode(ex))
    print(f"'{ex}' -> {n_tokens} tokens")

**Why this matters:** every LLM (including commercial ones like GPT-4 and Claude) charges and limits usage based on token count, not word count or character count. A model's context window (the maximum amount of text it can consider at once) is also measured in tokens. Understanding tokenization is the first step to understanding cost, limits, and behavior.

## Practice: Comparing Tokenization Across Languages

**Exercise:** Tokenize the same sentence written in English and a version with unusual or rare words (e.g., technical jargon, or a made-up word). Compare token counts and explain, in a comment, why the unusual text likely used more tokens per word.

In [ ]:
english_sentence = "I am learning about artificial intelligence."
unusual_sentence = "The mitochondria's electrochemical gradient drives ATP synthase."

### YOUR CODE HERE ###
for s in [english_sentence, unusual_sentence]:
    n_tokens = len(tokenizer.encode(s))
    n_words = len(s.split())
    print(f"{n_tokens} tokens, {n_words} words ({n_tokens / n_words:.2f} tokens/word): {tokenizer.tokenize(s)}")

# The unusual sentence uses more tokens per word because rare technical words
# (mitochondria, electrochemical, synthase) are not single entries in the
# vocabulary, so they are split into several sub-word pieces. Common words like
# "learning" or "about" are usually a single token.
### END ###

## Generating Text with a Language Model

Now let's load an actual small language model (`distilgpt2`, a lightweight version of GPT-2) and use it to generate text. This demonstrates the core LLM behavior: given a prompt, predict and generate the next tokens, one at a time.

In [ ]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained("distilgpt2")

prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

output = model.generate(
    input_ids,
    max_new_tokens=30,
    do_sample=False   # deterministic: always picks the most likely next token
)

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

This is a small, older, open-source model, so the output will be far less coherent than ChatGPT or Claude — but the underlying mechanism (predict the next token, repeat) is exactly the same one powering today's most advanced models, just at a vastly larger scale with much more training data.

## Sampling Parameters: Temperature & Top-p

When a model generates text, at each step it doesn't just have one "correct" next token — it has a probability distribution over its entire vocabulary. How the model picks from that distribution is controlled by sampling parameters:

- **Temperature**: controls randomness. Low temperature (e.g., 0.2) makes the model pick high-probability tokens almost every time, producing safe, repetitive, deterministic text. High temperature (e.g., 1.2) flattens the probability distribution, making less likely tokens more likely to be picked, producing more varied and creative (but also less coherent) text.
- **Top-p (nucleus sampling)**: instead of considering the entire vocabulary, the model only samples from the smallest set of tokens whose cumulative probability reaches p (e.g., 0.9). This cuts off very unlikely tokens while still allowing some randomness.

Let's compare outputs at different temperatures using the same prompt.

In [ ]:
prompt = "In the next ten years, technology will"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

for temp in [0.3, 0.7, 1.2]:
    torch.manual_seed(42)
    output = model.generate(
        input_ids,
        max_new_tokens=25,
        do_sample=True,
        temperature=temp,
        top_p=0.9
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    print(f"Temperature = {temp}:")
    print(text)
    print("-" * 60)

## Practice: Exploring Temperature Trade-offs

**Exercise:** Using the same prompt below, generate text at temperature=0.1 and temperature=1.5. In a markdown or comment, describe the difference in coherence and creativity you observe, and explain which setting you would use for (a) writing factual summaries and (b) writing creative fiction, and why.

In [ ]:
prompt = "The most important lesson in data science is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

### YOUR CODE HERE ###
for temp in [0.1, 1.5]:
    torch.manual_seed(42)
    output = model.generate(input_ids, max_new_tokens=30, do_sample=True,
                            temperature=temp, top_p=0.9)
    print(f"Temperature = {temp}:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))
    print("-" * 60)
### END ###

**Observation:** At temperature 0.1 the output is more coherent and predictable, and often repetitive, because the model almost always picks the most likely token. At 1.5 the output is more varied and surprising but is less coherent and may drift off topic or contain odd word choices. (Confirm this against your own output.)

**(a) Factual summaries:** use a low temperature, because accuracy and consistency matter more than variety.

**(b) Creative fiction:** use a higher temperature (moderately high, not extreme), because more varied word choices give more original writing, while still keeping the text readable.

## Context Windows

A model's context window is the maximum number of tokens it can process at once (input + output combined). If you exceed it, older text gets truncated or the request fails, depending on the system. For example, GPT-2 has a context window of 1024 tokens; modern models like Claude have context windows in the hundreds of thousands of tokens.

**Exercise:** Check GPT-2's maximum context length using the tokenizer's configuration, then calculate how many tokens are left available for generation if a prompt uses 900 tokens.

In [ ]:
max_context = tokenizer.model_max_length
prompt_tokens = 900

remaining = None

### YOUR CODE HERE ###
remaining = max_context - prompt_tokens
### END ###

print(f"Model max context: {max_context}")
print(f"Remaining tokens available for generation: {remaining}")

## Limitations of LLMs

Understanding what LLMs cannot reliably do is just as important as understanding what they can do:

- **Hallucination**: because LLMs generate text based on learned patterns rather than looking up verified facts, they can produce fluent, confident, and completely incorrect statements — especially about specific facts, dates, or citations.
- **Knowledge cutoff**: a model only knows about events up to whenever its training data was collected. It cannot know about anything that happened afterward unless that information is provided to it directly (this is the motivation for RAG, which you'll cover in Lab 3).
- **No true reasoning guarantee**: LLMs can produce reasoning-like text, but this is pattern generation, not guaranteed logical deduction — they can make basic arithmetic or logic errors that a calculator or simple program would never make.
- **Bias**: since models learn from large amounts of internet text, they can reproduce and sometimes amplify biases present in that training data.

We will see a concrete example of hallucination in the next cell.

In [ ]:
prompt = "The capital of the fictional country Wakanda, according to the 2023 census, is"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

output = model.generate(input_ids, max_new_tokens=20, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

Notice that the model doesn't say "I don't know" or "this is fictional" — it confidently continues generating plausible-sounding text regardless of whether the underlying premise is real. This is hallucination in action, and it's a core reason why LLM outputs always need some form of verification in real applications.

**What to remember from this lab:**
- An LLM is fundamentally a next-token predictor trained at massive scale
- Text is processed as tokens, not words — this affects cost, limits, and behavior
- Sampling parameters (temperature, top-p) control the randomness/creativity of output
- Every model has a fixed context window measured in tokens
- LLMs can hallucinate confidently — they don't know when they don't know something

## Lab Tasks

Complete the following tasks in this notebook, below this cell.

1. **Tokenization Analysis**: Choose three sentences of your own (one very simple, one technical/jargon-heavy, one containing a made-up word or name). Tokenize all three, print the token counts, and write 2-3 sentences explaining the pattern you observe.
2. **Temperature Comparison**: Using the same prompt, generate text at three different temperatures (your choice, spanning low to high) with `do_sample=True`. Print all three outputs, then write a short comparison of their coherence and creativity.
3. **Context Window Math**: If a model has a context window of 8,192 tokens, and a conversation so far has used 6,500 tokens, calculate how many tokens remain. Then, using the average estimate of ~0.75 words per token, estimate roughly how many words of new content this leaves room for.
4. **Hallucination Test**: Write your own prompt designed to test whether the model will hallucinate (e.g., ask about a fictional event, a very recent event, or a specific detailed fact it's unlikely to know precisely). Generate the output and analyze whether the model hallucinated, and how confidently it did so.
5. **Reflection**: In a markdown cell, explain in your own words why understanding tokenization and context windows matters for someone building a real application on top of an LLM (not just chatting with one casually).

In [ ]:
# Task 1
sentences = [
    "I like to read books.",
    "Gradient descent minimizes the cross-entropy loss via backpropagation.",
    "Zorblaxian quindrelfs glimmerate over the Plonkwhistle.",
]
for s in sentences:
    n_tokens = len(tokenizer.encode(s))
    n_words = len(s.split())
    print(f"{n_tokens} tokens / {n_words} words: {s}")
    print("   ", tokenizer.tokenize(s))

**Task 1 – Pattern:** The simple sentence uses roughly one token per word, since common words are single entries in the vocabulary. The technical sentence uses noticeably more tokens per word, and the made-up words use the most, because unfamiliar words are broken into many small sub-word pieces. This shows that token count depends on how common the text is in the training data, not just on its length. (Check these against your printed counts.)

In [ ]:
# Task 2
prompt = "In the future, people will"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

for temp in [0.2, 0.8, 1.5]:
    torch.manual_seed(42)
    output = model.generate(input_ids, max_new_tokens=30, do_sample=True, temperature=temp)
    print(f"Temperature = {temp}:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))
    print("-" * 60)

**Task 2 – Comparison:** At 0.2 the text is the most coherent but plain and repetitive. At 0.8 it is a good balance of fluency and variety. At 1.5 it is the most creative and unpredictable but often incoherent or ungrammatical. Higher temperature flattens the probability distribution, so less likely tokens get picked more often. (Adjust to match your outputs.)

In [ ]:
# Task 3
context_window = 8192
used = 6500
remaining_tokens = context_window - used
approx_words = remaining_tokens * 0.75

print("Remaining tokens:", remaining_tokens)
print("Approximate words of new content:", round(approx_words))

In [ ]:
# Task 4
prompt = "The 1987 Nobel Prize in Physics was awarded to Dr. Marcus Feldmoor for his discovery of"
input_ids = tokenizer.encode(prompt, return_tensors="pt")

output = model.generate(input_ids, max_new_tokens=30, do_sample=False)
print(tokenizer.decode(output[0], skip_special_tokens=True))

**Task 4 – Analysis:** "Dr. Marcus Feldmoor" is a made-up person, so no real discovery exists. The model still continues with a confident, fluent sentence describing a discovery instead of saying it does not know, which is a hallucination. It has no way to check facts and only predicts likely next tokens, and its output sounds equally sure whether or not the content is true. (Confirm this against your own output.)

**Task 5 – Reflection:** When building a real application, tokens are what you pay for and what limits you: API costs and speed scale with token count, so you need to estimate and control how many tokens each request uses. The context window caps how much the model can see at once, so long documents or chat histories must be truncated, summarized or retrieved selectively, and if you ignore the limit, requests fail or important earlier information gets lost. Tokenization also explains why unusual text (code, other languages, jargon) costs more and behaves differently than plain English. Understanding both lets a developer design prompts, budgets and data pipelines that are reliable and affordable instead of discovering these limits in production.